# Preparación de Datos para Modelado
## Temperatura Global — Pipeline con Orientación a Objetos

---

**Objetivo:** Aplicar el pipeline definido en `temperature_processor.py` para transformar los datos del EDA en un formato listo para entrenar un modelo de machine learning. Cada clase encapsula una etapa del proceso.

**Flujo del pipeline:**

```
CSV  →  [TemperatureLoader]  →  [TemperaturePreprocessor]  →  [TemperatureFeatureEngineer]  →  [TemperatureScaler]  →  X_train / X_test
```

**Principios OOP aplicados:**
- **Encapsulamiento**: cada clase tiene una responsabilidad única (Single Responsibility)
- **Composición**: `TemperaturePipeline` coordina todas las clases sin heredar de ellas
- **Interfaz consistente**: todas las clases transformadoras exponen `fit_transform(df)`

## 0. Importaciones

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# Importamos las clases definidas en nuestro módulo
from temperature_processor import (
    TemperatureLoader,
    TemperaturePreprocessor,
    TemperatureFeatureEngineer,
    TemperatureScaler,
    TemperaturePipeline,
)

plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_theme(style='whitegrid', palette='muted')

DATA_PATH = 'data/Global Temperatures.csv'
print('Módulo importado correctamente.')

Módulo importado correctamente.


---
## 1. Paso a paso: usando cada clase individualmente

Antes de usar el pipeline completo, recorremos cada etapa de forma explícita para entender qué hace cada clase.

### 1.1 `TemperatureLoader` — Carga y validación

In [2]:
loader = TemperatureLoader(DATA_PATH)
df_raw = loader.load()

print(f'Columnas normalizadas: {df_raw.columns.tolist()}')
print(f'Forma: {df_raw.shape}')
df_raw.head(3)

Columnas normalizadas: ['year', 'month', 'anomaly_monthly', 'unc_monthly', 'anomaly_annual', 'unc_annual', 'anomaly_5yr', 'unc_5yr', 'anomaly_10yr', 'unc_10yr', 'anomaly_20yr', 'unc_20yr']
Forma: (2077, 12)


,year,month,anomaly_monthly,unc_monthly,anomaly_annual,unc_annual,anomaly_5yr,unc_5yr,anomaly_10yr,unc_10yr,anomaly_20yr,unc_20yr
0,1850,1,-0.801,0.482,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1850,2,-0.102,0.592,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1850,3,-0.119,0.819,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 1.2 `TemperaturePreprocessor` — Limpieza y fechas

In [3]:
preprocessor = TemperaturePreprocessor()
df_clean = preprocessor.fit_transform(df_raw)

print('Resumen de nulos después del preprocesamiento:')
display(preprocessor.get_null_summary(df_clean))

print('\nColumnas nuevas creadas:')
nuevas = ['temp_abs', 'decade']
df_clean[nuevas].head(3)

Resumen de nulos después del preprocesamiento:


,nulos,porcentaje
year,0,0.00
month,0,0.00
anomaly_monthly,0,0.00
unc_monthly,13,0.63
anomaly_annual,11,0.53
unc_annual,24,1.16
anomaly_5yr,59,2.84
unc_5yr,72,3.47
anomaly_10yr,119,5.73
unc_10yr,132,6.36



Columnas nuevas creadas:


,temp_abs,decade
date,,
1850-01-01,11.509,1850
1850-02-01,12.418,1850
1850-03-01,13.031,1850


### 1.3 `TemperatureFeatureEngineer` — Construcción de features

Aquí creamos las variables de entrada (*features*) que usará el modelo.

In [ ]:
fe = TemperatureFeatureEngineer(
    lag_steps=[1, 2, 3, 6, 12],
    rolling_windows=[12, 60, 120]
)
df_features = fe.fit_transform(df_clean)

print(f'Features generadas ({len(fe.feature_columns)} columnas):')
for col in fe.feature_columns:
    print(f'  {col}')

In [ ]:
df_features[fe.feature_columns].dropna().head(5)

#### Visualizando las features de lag y media móvil

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Codificación cíclica del mes
meses = np.arange(1, 13)
sin_vals = np.sin(2 * np.pi * meses / 12)
cos_vals = np.cos(2 * np.pi * meses / 12)

MESES = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
axes[0].plot(meses, sin_vals, 'o-', label='month_sin', color='steelblue')
axes[0].plot(meses, cos_vals, 's-', label='month_cos', color='crimson')
axes[0].set_xticks(meses)
axes[0].set_xticklabels(MESES)
axes[0].set_title('Codificación Cíclica del Mes (sin/cos)')
axes[0].set_ylabel('Valor')
axes[0].legend()

# Comparación anomalía vs. lag-12
sample = df_features.dropna(subset=['anomaly_monthly', 'anomaly_lag_12']).iloc[-120:]
axes[1].plot(sample.index, sample['anomaly_monthly'], label='Anomalía mensual', alpha=0.7)
axes[1].plot(sample.index, sample['anomaly_lag_12'], label='Lag 12 meses (1 año atrás)',
             linestyle='--', alpha=0.7)
axes[1].set_title('Feature lag-12: anomalía vs. valor del año anterior')
axes[1].set_ylabel('Anomalía (°C)')
axes[1].legend()

plt.tight_layout()
plt.show()

### 1.4 `TemperatureScaler` — Estandarización

**¿Por qué estandarizar?** Muchos modelos (regresión lineal, redes neuronales, SVM) son sensibles a la escala de las features. Estandarizar lleva todas las variables a la misma escala (media 0, desviación estándar 1) sin distorsionar la distribución.

In [ ]:
df_no_nulos = df_features.dropna(subset=fe.feature_columns + ['anomaly_monthly'])

scaler = TemperatureScaler()
df_scaled = scaler.fit_transform(df_no_nulos, columns=fe.feature_columns)

print('Parámetros del escalador (media y desv. std. calculadas sobre train):')
display(scaler.params.round(4))

In [ ]:
# Verificación: media ≈ 0 y std ≈ 1 en el conjunto escalado
print('Estadísticos del conjunto escalado (deben ser ≈ media=0, std=1):')
df_scaled[fe.feature_columns].agg(['mean', 'std']).round(4)

---
## 2. Pipeline completo con `TemperaturePipeline`

Ahora usamos la clase que orquesta todo el proceso en un solo `run()`.

In [ ]:
pipeline = TemperaturePipeline(
    filepath=DATA_PATH,
    test_year_start=2000,
    lag_steps=[1, 2, 3, 6, 12],
    rolling_windows=[12, 60, 120]
)

splits = pipeline.run()

In [ ]:
X_train = splits['X_train']
X_test  = splits['X_test']
y_train = splits['y_train']
y_test  = splits['y_test']

print('X_train shape:', X_train.shape)
print('X_test shape: ', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape: ', y_test.shape)

print('\nPrimeras filas de X_train:')
X_train.head()

---
## 3. División temporal Train / Test

**¿Por qué usar división temporal y no aleatoria?**  
En series de tiempo, el orden importa. Si mezclamos aleatoriamente, el modelo podría "ver el futuro" durante el entrenamiento, lo que inflaría artificialmente su rendimiento. La división temporal garantiza que el modelo solo aprende con datos pasados y se evalúa con datos futuros.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))

ax.fill_between(y_train.index, y_train, 0,
                where=(y_train >= 0), alpha=0.4, color='steelblue')
ax.fill_between(y_train.index, y_train, 0,
                where=(y_train < 0), alpha=0.4, color='royalblue')

ax.fill_between(y_test.index, y_test, 0,
                where=(y_test >= 0), alpha=0.4, color='tomato')
ax.fill_between(y_test.index, y_test, 0,
                where=(y_test < 0), alpha=0.4, color='salmon')

ax.axvline(y_test.index.min(), color='black', linestyle='--', linewidth=1.5,
           label=f'Corte train/test: {pipeline.test_year_start}')

# Leyenda manual
from matplotlib.patches import Patch
leg = [
    Patch(color='steelblue', alpha=0.5, label=f'Train ({y_train.index.min().year}–{y_train.index.max().year})'),
    Patch(color='tomato', alpha=0.5, label=f'Test ({y_test.index.min().year}–{y_test.index.max().year})'),
]
ax.legend(handles=leg + [plt.Line2D([0],[0], color='black', linestyle='--', label='Corte')])
ax.axhline(0, color='black', linewidth=0.7)
ax.set_title('División Temporal Train / Test (Target: anomalía mensual)')
ax.set_xlabel('Año')
ax.set_ylabel('Anomalía (°C)')
plt.tight_layout()
plt.show()

---
## 4. Análisis de Features para el Modelo

### 4.1 Correlación de cada feature con el target

In [ ]:
df_train_full = pd.concat([X_train, y_train], axis=1)

corr_con_target = df_train_full.corr()['anomaly_monthly'].drop('anomaly_monthly').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['crimson' if c >= 0 else 'steelblue' for c in corr_con_target]
ax.barh(corr_con_target.index, corr_con_target.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlación de cada Feature con el Target (anomaly_monthly)')
ax.set_xlabel('Correlación de Pearson')
plt.tight_layout()
plt.show()

print('Top 5 features más correlacionadas con el target:')
print(corr_con_target.head(5).round(4))

### 4.2 Distribución de las features escaladas

In [ ]:
n_cols = 4
feature_list = X_train.columns.tolist()
n_rows = -(-len(feature_list) // n_cols)  # techo de la división

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3))
axes = axes.flatten()

for i, col in enumerate(feature_list):
    sns.histplot(X_train[col].dropna(), bins=30, ax=axes[i],
                 color='steelblue', kde=True)
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel('')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distribución de Features Escaladas (X_train)', y=1.01)
plt.tight_layout()
plt.show()

---
## 5. Ejemplo de modelo: Regresión Lineal con NumPy

Implementamos un modelo de regresión lineal múltiple usando solo NumPy para ilustrar el concepto. En producción usaríamos scikit-learn u otra librería.

**Modelo:** `y = X · w + b`  
**Ajuste:** Mínimos cuadrados ordinarios (OLS)

In [ ]:
# Preparar matrices NumPy eliminando NaN residuales
mask_train = X_train.notna().all(axis=1) & y_train.notna()
mask_test  = X_test.notna().all(axis=1)  & y_test.notna()

Xtr = X_train[mask_train].values
ytr = y_train[mask_train].values
Xte = X_test[mask_test].values
yte = y_test[mask_test].values

# Agregar columna de unos para el intercepto (bias)
Xtr_b = np.column_stack([np.ones(len(Xtr)), Xtr])
Xte_b = np.column_stack([np.ones(len(Xte)), Xte])

# Solución analítica de OLS:  w = (X^T X)^{-1} X^T y
w = np.linalg.lstsq(Xtr_b, ytr, rcond=None)[0]

# Predicciones
y_pred_train = Xtr_b @ w
y_pred_test  = Xte_b @ w

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1 - ss_res / ss_tot

print(f'  RMSE Train : {rmse(ytr, y_pred_train):.4f} °C')
print(f'  RMSE Test  : {rmse(yte, y_pred_test):.4f} °C')
print(f'  R²   Train : {r2(ytr, y_pred_train):.4f}')
print(f'  R²   Test  : {r2(yte, y_pred_test):.4f}')

In [ ]:
# Visualización: real vs. predicción en el set de prueba
fechas_test = y_test[mask_test].index

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=False)

# Serie temporal
axes[0].plot(fechas_test, yte, label='Real', color='steelblue', linewidth=1.5)
axes[0].plot(fechas_test, y_pred_test, label='Predicción (OLS)',
             color='crimson', linewidth=1.5, linestyle='--')
axes[0].set_title('Anomalía Real vs. Predicción — Conjunto de Prueba')
axes[0].set_ylabel('Anomalía (°C)')
axes[0].legend()

# Scatter real vs. predicción
axes[1].scatter(yte, y_pred_test, s=15, alpha=0.5, color='steelblue')
lims = [min(yte.min(), y_pred_test.min()) - 0.1,
        max(yte.max(), y_pred_test.max()) + 0.1]
axes[1].plot(lims, lims, 'k--', linewidth=1, label='Predicción perfecta (y=x)')
axes[1].set_xlabel('Valor Real (°C)')
axes[1].set_ylabel('Predicción (°C)')
axes[1].set_title(f'Scatter: Real vs. Predicción  (R² = {r2(yte, y_pred_test):.4f})')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 6. Resumen del módulo `temperature_processor.py`

```
temperature_processor.py
│
├── TemperatureLoader
│   ├── __init__(filepath)
│   ├── load() → DataFrame
│   └── _validate()              # verificación de estructura
│
├── TemperaturePreprocessor
│   ├── fit_transform(df) → DataFrame
│   ├── _build_date_index(df)
│   ├── _add_absolute_temperature(df)
│   ├── _add_decade(df)
│   └── get_null_summary(df)     # utilidad de diagnóstico
│
├── TemperatureFeatureEngineer
│   ├── __init__(lag_steps, rolling_windows)
│   ├── fit_transform(df) → DataFrame
│   ├── _encode_month_cyclic(df) # sin/cos del mes
│   ├── _add_lag_features(df)    # valores pasados del target
│   ├── _add_rolling_features(df)# medias móviles
│   ├── _add_linear_trend(df)    # tendencia [0,1]
│   └── feature_columns (propiedad)
│
├── TemperatureScaler
│   ├── fit(df, columns)
│   ├── transform(df) → DataFrame
│   ├── fit_transform(df, columns) → DataFrame
│   ├── inverse_transform_column(col, values)
│   └── params (propiedad) → DataFrame
│
└── TemperaturePipeline
    ├── __init__(filepath, test_year_start, lag_steps, rolling_windows)
    ├── run() → dict{X_train, X_test, y_train, y_test, df_full}
    ├── processed (propiedad)
    └── features (propiedad)
```

### Próximos modelos a explorar

Con los datos listos (`X_train`, `X_test`, `y_train`, `y_test`) se puede entrenar cualquiera de estos modelos:

| Modelo | Librería | Uso sugerido |
|--------|----------|--------------|
| Regresión Lineal/Ridge | `sklearn.linear_model` | Baseline simple |
| Random Forest | `sklearn.ensemble` | Captura no-linealidades |
| XGBoost / LightGBM | `xgboost`, `lightgbm` | Alta precisión con features tabulares |
| ARIMA / SARIMA | `statsmodels` | Modelos estadísticos de series de tiempo |
| Prophet | `prophet` | Tendencias + estacionalidad automática |
| LSTM | `tensorflow/keras` | Patrones de largo plazo |
